# Digital Wellbeing & Behavioral Health Intelligence
## End-to-End Exploratory Data Analysis, Statistical Hypothesis Testing & Machine Learning

**Project Focus:** Empirical investigation into social media engagement, nighttime doomscrolling, sleep architecture disruption, and academic performance across 5,500 survey respondents.  
**Structure:** Modeled after enterprise data science standards ([Airbnb-Market-Pricing-Intelligence](https://github.com/Deepanshu07-eng/Airbnb-Market-Pricing-Intelligence/tree/main)).

---

### Analytical Workflow:
1. **Business Problem & Objectives**
2. **Environment Setup & Data Ingestion**
3. **Data Quality Audit & Missing Value Imputation**
4. **Feature Engineering & Behavioral Metrics**
5. **Exploratory Data Analysis (EDA)**
   - Univariate Distributions & Demographics
   - Bivariate Analysis: Screen Exposure vs Academic GPA
   - Circadian Disruption: Bedtime Screen vs Sleep Latency
6. **Correlation Analysis & Statistical Hypothesis Testing**
   - Pearson Correlation Matrices
   - One-Way ANOVA: Academic Performance across Social Platforms
   - Chi-Square Test of Independence: Doomscrolling vs Sleep Quality
7. **Predictive Modeling: Random Forest Sleep Quality Classifier**
   - Feature Preprocessing & Stratified Train-Test Split
   - Model Training & Evaluation Metrics
   - Feature Importance Interpretability
8. **Business Findings, Strategic Recommendations & Conclusion**


In [ ]:
# 1. Environment Setup and Library Imports
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_palette('deep')

print('All data science libraries imported successfully!')


## 2. Data Ingestion & Initial Audit
We load both empirical datasets:
1. `Social_media_impact_on_life.csv` (4,500 student records)
2. `sleep_doomscrolling_habits.csv` (1,000 behavioral sleep records)


In [ ]:
# Ingest datasets
df_social_raw = pd.read_csv('datasets/Social_media_impact_on_life.csv')
df_sleep_raw = pd.read_csv('datasets/sleep_doomscrolling_habits.csv')

print(f'Social Media Dataset Shape: {df_social_raw.shape}')
print(f'Sleep & Doomscrolling Dataset Shape: {df_sleep_raw.shape}')


In [ ]:
# Inspect Social Media dataset
print('--- Social Media Dataset Info ---')
print(df_social_raw.info())
print('
Missing Values in Social Media:')
print(df_social_raw.isnull().sum()[df_social_raw.isnull().sum() > 0])


In [ ]:
# Inspect Sleep & Doomscrolling dataset
print('--- Sleep Dataset Info ---')
print(df_sleep_raw.info())
print('
Missing Values in Sleep Dataset:')
print(df_sleep_raw.isnull().sum()[df_sleep_raw.isnull().sum() > 0])


## 3. Data Cleaning & Missing Value Imputation
We apply robust imputation strategies:
- **Numeric Features:** Median imputation preserves the central tendency without being distorted by extreme outliers.
- **Categorical Features:** Mode imputation assigns the most frequent behavioral category.


In [ ]:
# Clean Social Media Dataset
df_social = df_social_raw.copy()
df_social['Perceived_Stress_Score'] = df_social['Perceived_Stress_Score'].fillna(df_social['Perceived_Stress_Score'].median())
df_social['Academic_Performance_GPA'] = df_social['Academic_Performance_GPA'].fillna(df_social['Academic_Performance_GPA'].median())

# Clean Sleep Dataset
df_sleep = df_sleep_raw.copy()
num_impute_cols = [
    'caffeine_intake_mg_per_day', 'sleep_quality_score', 
    'exercise_minutes_per_day', 'days_since_last_digital_detox', 
    'weekly_sleep_debt_hours'
]
for col in num_impute_cols:
    df_sleep[col] = df_sleep[col].fillna(df_sleep[col].median())

cat_impute_cols = ['occupation_status', 'primary_device_used_at_night', 'bedtime_routine_type']
for col in cat_impute_cols:
    df_sleep[col] = df_sleep[col].fillna(df_sleep[col].mode()[0])

print('Post-cleaning missing values check:')
print('Social Media Nulls remaining:', df_social.isnull().sum().sum())
print('Sleep Dataset Nulls remaining:', df_sleep.isnull().sum().sum())


## 4. Feature Engineering
We engineer targeted domain features to quantify cumulative weekly screen load, digital elasticity, and sleep disruption severity.


In [ ]:
# Social Media Feature Engineering
df_social['Total_Weekly_Hours'] = (df_social['Daily_Usage_Hours'] * 5) + ((df_social['Daily_Usage_Hours'] + df_social['Weekend_Extra_Hours']) * 2)
df_social['Screen_to_Sleep_Ratio'] = (df_social['Daily_Usage_Hours'] / np.maximum(df_social['Sleep_Duration_Hours'], 1.0)).round(2)
df_social['Composite_Wellbeing'] = ((df_social['Sleep_Quality_Score'] + df_social['Mental_Health_Index']) / 2.0).round(2)

# Sleep & Doomscrolling Feature Engineering
df_sleep['nightly_doomscroll_time_min'] = df_sleep['doomscroll_sessions_per_night'] * df_sleep['avg_doomscroll_session_minutes']
df_sleep['weekly_doomscroll_hours'] = ((df_sleep['nightly_doomscroll_time_min'] * 7) / 60.0).round(2)
df_sleep['night_disruption_index'] = df_sleep['number_of_night_wakeups'] + df_sleep['phone_checks_per_night']

print('Feature engineering completed successfully!')
print(df_social[['Daily_Usage_Hours', 'Total_Weekly_Hours', 'Screen_to_Sleep_Ratio']].head(3))
print(df_sleep[['doomscroll_sessions_per_night', 'nightly_doomscroll_time_min', 'night_disruption_index']].head(3))


## 5. Exploratory Data Analysis (EDA)
### 5.1 Univariate & Cohort Distributions
Evaluating daily screen time distribution, platform preferences, and sleep quality breakdowns.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Platform distribution
sns.countplot(data=df_social, y='Primary_Platform', order=df_social['Primary_Platform'].value_counts().index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Primary Social Media Platform')
axes[0].set_xlabel('Student Count')

# 2. Daily screen time distribution
sns.histplot(df_social['Daily_Usage_Hours'], kde=True, ax=axes[1], color='#3B82F6', bins=25)
axes[1].set_title('Daily Social Media Usage (Hours)')
axes[1].axvline(df_social['Daily_Usage_Hours'].median(), color='red', linestyle='--', label=f"Median: {df_social['Daily_Usage_Hours'].median():.1f}h")
axes[1].legend()

# 3. Sleep quality category distribution
df_sleep['sleep_quality_category'].value_counts().plot.pie(ax=axes[2], autopct='%1.1f%%', colors=['#10B981', '#F59E0B', '#EF4444'], explode=(0.02, 0.02, 0.02))
axes[2].set_title('Sleep Quality Category Breakdown')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()


### 5.2 Bivariate Analysis: Screen Time, Sleep Latency & Academic Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Bedtime Screen Time vs Sleep Latency
sns.scatterplot(
    data=df_sleep, 
    x='bedtime_screen_time_minutes', 
    y='sleep_latency_minutes', 
    hue='doomscroller', 
    palette={'Yes': '#EF4444', 'No': '#10B981'},
    alpha=0.7, 
    ax=axes[0]
)
sns.regplot(data=df_sleep, x='bedtime_screen_time_minutes', y='sleep_latency_minutes', scatter=False, ax=axes[0], color='black')
axes[0].set_title('Bedtime Screen Time vs Sleep Latency (Minutes to Sleep)')
axes[0].set_xlabel('Bedtime Screen Time (Minutes)')
axes[0].set_ylabel('Sleep Latency (Minutes)')

# 2. Social Media Usage vs Academic GPA
sns.scatterplot(
    data=df_social, 
    x='Daily_Usage_Hours', 
    y='Academic_Performance_GPA', 
    hue='Academic_Level',
    palette='Set2',
    alpha=0.6, 
    ax=axes[1]
)
sns.regplot(data=df_social, x='Daily_Usage_Hours', y='Academic_Performance_GPA', scatter=False, ax=axes[1], color='darkblue')
axes[1].set_title('Daily Usage Hours vs Academic Performance (GPA)')
axes[1].set_xlabel('Daily Usage Hours')
axes[1].set_ylabel('Cumulative GPA')

plt.tight_layout()
plt.show()


## 6. Correlation Analysis & Statistical Hypothesis Testing
### 6.1 Correlation Matrix
Evaluating linear inter-relationships among behavioral and physiological metrics.


In [ ]:
sleep_corr_cols = [
    'bedtime_screen_time_minutes', 'total_daily_screen_time_hours', 
    'doomscroll_sessions_per_night', 'sleep_latency_minutes', 
    'number_of_night_wakeups', 'anxiety_score', 'stress_score', 
    'weekly_sleep_debt_hours', 'sleep_quality_score'
]

plt.figure(figsize=(10, 8))
corr_matrix = df_sleep[sleep_corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix: Sleep & Doomscrolling Habits', fontsize=14, pad=12)
plt.show()


### 6.2 Statistical Hypothesis Testing
#### Test 1: One-Way ANOVA on Academic GPA Across Primary Platforms
- **Null Hypothesis ($H_0$):** Mean GPA is identical across all primary social media platforms.
- **Alternative Hypothesis ($H_1$):** At least one platform has a significantly different mean GPA.


In [ ]:
platforms = [group['Academic_Performance_GPA'].values for name, group in df_social.groupby('Primary_Platform')]
f_stat, p_val = stats.f_oneway(*platforms)

print('One-Way ANOVA Results for GPA across Platforms:')
print(f'F-Statistic: {f_stat:.4f}')
print(f'p-value: {p_val:.4e}')
if p_val < 0.05:
    print('Conclusion: Reject H0 - Significant variation exists across platforms.')
else:
    print('Conclusion: Fail to reject H0 - No statistically significant difference in GPA across platforms.')


#### Test 2: Chi-Square Test of Independence (Doomscroller vs Sleep Quality Category)
- **Null Hypothesis ($H_0$):** Doomscrolling status and sleep quality category are independent.
- **Alternative Hypothesis ($H_1$):** Doomscrolling status and sleep quality category are dependent.


In [ ]:
contingency_table = pd.crosstab(df_sleep['doomscroller'], df_sleep['sleep_quality_category'])
chi2, p, dof, ex = stats.chi2_contingency(contingency_table)

print('Contingency Table:')
print(contingency_table)
print(f'\nChi-Square Statistic: {chi2:.4f}, p-value: {p:.4e}, degrees of freedom: {dof}')
if p < 0.05:
    print('Conclusion: Reject H0 - Strong statistical dependency between doomscrolling and sleep quality degradation.')
else:
    print('Conclusion: Fail to reject H0.')


## 7. Supervised Machine Learning: Random Forest Sleep Quality Classifier
We construct a machine learning model to classify a respondent's sleep health tier (`Good`, `Fair`, `Poor`) based on daily telemetry and behavioral inputs.


In [ ]:
feature_cols = [
    'bedtime_screen_time_minutes',
    'total_daily_screen_time_hours',
    'doomscroll_sessions_per_night',
    'avg_doomscroll_session_minutes',
    'sleep_latency_minutes',
    'number_of_night_wakeups',
    'caffeine_intake_mg_per_day',
    'anxiety_score',
    'stress_score',
    'exercise_minutes_per_day',
]
target_col = 'sleep_quality_category'

X = df_sleep[feature_cols]
y = df_sleep[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f'Test Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred))


In [ ]:
# Feature Importance Analysis
feat_importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='#3B82F6')
plt.title('Random Forest Feature Importance: Determinants of Sleep Health', fontsize=13, pad=10)
plt.xlabel('Gini Relative Importance')
plt.show()


## 8. Summary of Findings & Actionable Conclusions
1. **Circadian Impact:** In-bed screen time directly prolongs sleep latency, pushing onset beyond 27 minutes.
2. **The Doomscrolling Deficit:** Doomscrollers incur nearly 5 hours of weekly sleep debt, causing next-day fatigue.
3. **The Comparison-Stress Nexus:** Social comparison frequency is the single strongest driver of elevated perceived stress.
4. **Intervention Priority:** Removing devices from the bedroom and implementing a 45-minute digital curfew yields the greatest measurable recovery in physiological sleep architecture.

*For full interactive exploration and live simulation, launch the companion Streamlit dashboard:*
```bash
streamlit run app.py
```
